In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
df = pd.read_csv('../Data/Raw_FEMA_Data/FEMA_Declarations.csv')
 

In [3]:
df.head()
print(df.columns.tolist())

['femaDeclarationString', 'disasterNumber', 'state', 'declarationType', 'declarationDate', 'fyDeclared', 'incidentType', 'declarationTitle', 'ihProgramDeclared', 'iaProgramDeclared', 'paProgramDeclared', 'hmProgramDeclared', 'incidentBeginDate', 'incidentEndDate', 'disasterCloseoutDate', 'tribalRequest', 'fipsStateCode', 'fipsCountyCode', 'placeCode', 'designatedArea', 'declarationRequestNumber', 'lastIAFilingDate', 'incidentId', 'region', 'designatedIncidentTypes', 'lastRefresh', 'hash', 'id']


In [4]:
df.shape

(69706, 28)

#### Dropping extra or unneeded columns, starting off by defining those columns. 
#### In our SQL table (Based on our ERD) we will only need state, declarationType, incidentType, ihProgramDeclared, iaProgramDeclared, incidentBeginDate.

In [5]:
cols_to_drop = ['femaDeclarationString', 'disasterNumber', 'fyDeclared', 
                    'declarationTitle', 'placeCode', 'designatedArea', 'declarationRequestNumber', 
                    'lastIAFilingDate', 'incidentId', 'region', 'designatedIncidentTypes', 'lastRefresh',
                    'hash', 'id','paProgramDeclared', 'hmProgramDeclared',
                    'disasterCloseoutDate', 'tribalRequest', 'fipsStateCode', 'fipsCountyCode', 'declarationDate', 'incidentEndDate'
                    ]

FEMA_df = df.drop(columns=cols_to_drop, errors='ignore')

FEMA_df['incidentBeginDate'] = pd.to_datetime(FEMA_df['incidentBeginDate'])
FEMA_df['year'] = FEMA_df['incidentBeginDate'].dt.year
FEMA_df = FEMA_df.drop(columns=['incidentBeginDate'])
FEMA_df.head()

,state,declarationType,incidentType,ihProgramDeclared,iaProgramDeclared,year
0,OR,FM,Fire,0,0,2024
1,OR,FM,Fire,0,0,2024
2,OR,FM,Fire,0,0,2024
3,CA,DR,Severe Storm,0,0,2017
4,AL,DR,Severe Storm,0,0,2015


In [6]:
print(FEMA_df['incidentType'].unique())

['Fire' 'Severe Storm' 'Straight-Line Winds' 'Flood' 'Hurricane'
 'Biological' 'Winter Storm' 'Tornado' 'Tropical Storm' 'Earthquake'
 'Typhoon' 'Snowstorm' 'Freezing' 'Mud/Landslide' 'Coastal Storm' 'Other'
 'Severe Ice Storm' 'Dam/Levee Break' 'Volcanic Eruption'
 'Tropical Depression' 'Toxic Substances' 'Chemical' 'Terrorist' 'Drought'
 'Human Cause' 'Fishing Losses' 'Tsunami']


In [7]:
FEMA_df = FEMA_df[FEMA_df['incidentType'].isin([
    'Tornado',
    'Severe Storm',
    'Straight-Line Winds',
    'Coastal Storm'
])]

FEMA_df = FEMA_df[FEMA_df['year'].isin([2004, 2014, 2024])]

FEMA_df.head()

,state,declarationType,incidentType,ihProgramDeclared,iaProgramDeclared,year
595,MT,DR,Straight-Line Winds,0,0,2024
596,MT,DR,Straight-Line Winds,0,0,2024
608,OK,DR,Severe Storm,0,0,2024
718,NE,DR,Severe Storm,0,0,2024
719,NE,DR,Severe Storm,0,0,2024


##### For the sake of consistency, we will want to rename these columns to match the created columns in the SQL database. If we fail to do so, the will not import correctly into SQL.

In [9]:
FEMA_df = FEMA_df.rename(columns={
    'disasterNumber':   'disaster_number',
    'declarationType':  'declaration_type',
    'incidentType':     'incident_type',
    'ihProgramDeclared': 'ih_declared',
    'iaProgramDeclared': 'ia_declared'
})

FEMA_df.head()

,state,declaration_type,incident_type,ih_declared,ia_declared,year
595,MT,DR,Straight-Line Winds,0,0,2024
596,MT,DR,Straight-Line Winds,0,0,2024
608,OK,DR,Severe Storm,0,0,2024
718,NE,DR,Severe Storm,0,0,2024
719,NE,DR,Severe Storm,0,0,2024
